# Estimating Embedding Cost

This notebook demonstrates how to estimate the cost of generating embeddings with the OpenAI API **before** making any API calls.

**Model:** `text-embedding-3-small`  
**Cost:** $0.02 per 1M tokens  
**Token counter:** [`tiktoken`](https://github.com/openai/tiktoken) — OpenAI's fast tokenizer

The workflow:
1. Build a sample dataset of documents
2. Tokenize every document locally (no API call)
3. Sum the tokens and multiply by the per-token price

## Install / verify dependencies

`tiktoken` must be installed. If you set up the project via `requirements.txt` it is already present.

In [ ]:
import importlib, sys

if importlib.util.find_spec("tiktoken") is None:
    raise ImportError(
        "tiktoken is not installed. Run: pip install tiktoken"
    )

import tiktoken
print(f"tiktoken {tiktoken.__version__} ready")

## Sample dataset

A small Netflix-style catalogue. Each entry is formatted the same way it would be stored as a ChromaDB document — title, description, and categories on separate lines. This mirrors what you'd pass to `collection.add(documents=[...])`.

In [ ]:
raw_data = [
    {
        "title": "Kota Factory",
        "type": "TV Show",
        "description": "In a city of coaching centers known to train India's finest engineering minds, an earnest student navigates the pressures of competitive academics and adolescent life.",
        "categories": ["International TV Shows", "Romantic TV Shows", "TV Comedies"],
    },
    {
        "title": "The Last Letter From Your Lover",
        "type": "Movie",
        "description": "After finding a trove of love letters from 1965, a reporter sets out to discover the story of the couple who wrote them and uncover their mysterious ending.",
        "categories": ["Dramas", "Romantic Movies"],
    },
    {
        "title": "Squid Game",
        "type": "TV Show",
        "description": "Hundreds of cash-strapped players accept a strange invitation to compete in children's games. Inside, a tempting prize awaits — with deadly high stakes.",
        "categories": ["Korean TV Shows", "Thrillers"],
    },
    {
        "title": "The Power of the Dog",
        "type": "Movie",
        "description": "A domineering rancher responds with mocking cruelty when his brother brings home a new wife and her son, until the unexpected comes to pass.",
        "categories": ["Dramas", "Independent Movies"],
    },
    {
        "title": "Our Planet",
        "type": "TV Show",
        "description": "Experience our planet's natural beauty and examine how climate change impacts all living creatures in this ambitious documentary series.",
        "categories": ["Documentaries", "Nature & Science TV"],
    },
]

print(f"{len(raw_data)} items loaded")

## Format documents

Convert each record into a single string that will be embedded. The format matches how the data would be stored in ChromaDB.

In [ ]:
def format_document(item: dict) -> str:
    categories = ", ".join(item["categories"])
    return (
        f"Title: {item['title']} ({item['type']})\n"
        f"Description: {item['description']}\n"
        f"Categories: {categories}"
    )


documents = [format_document(item) for item in raw_data]

for doc in documents:
    print(doc)
    print()

## Count tokens with tiktoken

`tiktoken.encoding_for_model` returns the exact tokenizer OpenAI uses for a given model, so the count matches what the API will bill.

In [ ]:
MODEL = "text-embedding-3-small"

enc = tiktoken.encoding_for_model(MODEL)

token_counts = [len(enc.encode(doc)) for doc in documents]
total_tokens = sum(token_counts)

print(f"{'Document':<45} {'Tokens':>6}")
print("-" * 52)
for item, count in zip(raw_data, token_counts):
    label = f"{item['title']} ({item['type']})"
    print(f"{label:<45} {count:>6}")
print("-" * 52)
print(f"{'Total tokens':<45} {total_tokens:>6}")

## Estimate cost

`text-embedding-3-small` is priced at **$0.02 per 1,000,000 tokens** (see [OpenAI pricing](https://developers.openai.com/api/docs/pricing#embeddings)).

In [ ]:
COST_PER_1M_TOKENS = 0.02  # USD, text-embedding-3-small

cost = COST_PER_1M_TOKENS * total_tokens / 1000000

print(f"Model:        {MODEL}")
print(f"Total tokens: {total_tokens:,}")
print(f"Cost/1M tok:  ${COST_PER_1M_TOKENS}")
print(f"Estimated cost: ${cost:.6f}")

## Scale estimate

Use the per-document average to project cost across larger datasets — useful before committing to a full embedding run.

In [ ]:
avg_tokens = total_tokens / len(documents)

print(f"Average tokens per document: {avg_tokens:.1f}")
print()
print(f"{'Documents':>12}  {'Est. tokens':>14}  {'Est. cost (USD)':>16}")
print("-" * 48)
for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    est_tokens = avg_tokens * n
    est_cost = COST_PER_1M_TOKENS * est_tokens / 1000000
    print(f"{n:>12,}  {est_tokens:>14,.0f}  ${est_cost:>15.4f}")

## References

- [OpenAI Pricing](https://developers.openai.com/api/docs/pricing#embeddings) — Current rates for all embedding models
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings) — Model comparison and best practices
- [tiktoken on GitHub](https://github.com/openai/tiktoken) — Fast BPE tokenizer used by OpenAI models